# Sequence Generation for Deep Learning Forecasting

## Objective

This notebook converts the scaled M5 retail demand data into supervised
time-series sequences.

The forecasting task is formulated as:

> Use the previous 28 days of demand to predict the following day's demand.

Each input sequence therefore contains:

- 28 historical time steps
- 1 demand feature per time step

The resulting input representation has the shape:

    (samples, 28, 1)

The target has the shape:

    (samples,)

This representation is compatible with recurrent, convolutional and
attention-based deep learning architectures.

The same sequence configuration will be used for:

- LSTM
- GRU
- TCN
- Transformer

## Temporal Leakage Prevention

The target of each sequence is always later than every observation in
its input window.

For validation and test targets, historical observations from earlier
periods may be used as context because those observations would have
already occurred in a real forecasting scenario.

Future target observations are never used as historical inputs.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

if 'google.colab' in sys.modules:
    BASE = Path("/content")
else:
    BASE = Path("..").resolve()

sys.path.insert(0, str(BASE / "src"))

In [2]:
processed_path = BASE / "data" / "m5" / "processed"
results_path = BASE / "results"

results_path.mkdir(
    parents=True,
    exist_ok=True
)

In [3]:
sales_clean = pd.read_csv(
    processed_path / "sales_train_validation_clean.csv"
)

print("Sales shape:", sales_clean.shape)

Sales shape: (30490, 1919)


In [4]:
sales_columns = [
    column
    for column in sales_clean.columns
    if column.startswith("d_")
]

print("Number of sales days:", len(sales_columns))
print("First day:", sales_columns[0])
print("Last day:", sales_columns[-1])

Number of sales days: 1913
First day: d_1
Last day: d_1913


In [5]:
import importlib
import data_splitting

importlib.reload(data_splitting)

from data_splitting import create_chronological_split

split = create_chronological_split(
    sales_clean,
    train_ratio=0.80,
    validation_ratio=0.10
)

print("Training days:", split["train_days"])
print("Validation days:", split["validation_days"])
print("Test days:", split["test_days"])

Training days: 1530
Validation days: 191
Test days: 192


In [6]:
import scaling

importlib.reload(scaling)

from scaling import (
    fit_scaler_on_training_data,
    transform_sales_splits
)

scaler = fit_scaler_on_training_data(
    sales_clean,
    split["train"]
)

scaled_data = transform_sales_splits(
    sales_clean,
    split,
    scaler
)

print(
    "Train:",
    scaled_data["train"].shape
)

print(
    "Validation:",
    scaled_data["validation"].shape
)

print(
    "Test:",
    scaled_data["test"].shape
)

Train: (30490, 1530)
Validation: (30490, 191)
Test: (30490, 192)


## Sequence Boundary Design

The sequence generator operates on chronological target periods.

For a 28-day lookback:

    t-28 ... t-2 t-1 → t

The target day `t` is never included in its own input.

For example:

    Days 100–127 → predict Day 128

For a validation target at the beginning of the validation period, the
28-day historical context can contain days from the end of the training
period.

This is not leakage because those days occurred before the validation
prediction date.

The same principle applies to the test period: historical observations
from before the test target may be used as context, while test target
values are not used to create earlier predictions.

In [7]:
LOOKBACK = 28

print("Lookback window:", LOOKBACK, "days")

Lookback window: 28 days


In [8]:
scaled_all = np.concatenate(
    [
        scaled_data["train"],
        scaled_data["validation"],
        scaled_data["test"]
    ],
    axis=1
)

print("Combined scaled data shape:", scaled_all.shape)

assert scaled_all.shape == (
    len(sales_clean),
    len(sales_columns)
)

print("Combined scaled data validated.")

Combined scaled data shape: (30490, 1913)
Combined scaled data validated.


In [9]:
train_end = split["train_days"]

validation_end = (
    train_end +
    split["validation_days"]
)

test_end = (
    validation_end +
    split["test_days"]
)

print("Training target boundary:", train_end)
print("Validation target boundary:", validation_end)
print("Test target boundary:", test_end)

Training target boundary: 1530
Validation target boundary: 1721
Test target boundary: 1913


In [10]:
import sequence_generator

importlib.reload(sequence_generator)

from sequence_generator import (
    generate_sequences_with_global_indices,
    convert_to_tensors,
    validate_sequence_shapes,
    check_sequence_leakage,
    RetailSequenceDataset
)

In [11]:
train_dataset = RetailSequenceDataset(
    data=scaled_all,
    start_target=LOOKBACK,
    end_target=train_end,
    lookback=LOOKBACK
)

print("Training sequences:", len(train_dataset))

expected_train_sequences = (
    len(sales_clean) *
    (train_end - LOOKBACK)
)

print(
    "Expected training sequences:",
    expected_train_sequences
)

print(
    "Actual training sequences:",
    len(train_dataset)
)

assert len(train_dataset) == expected_train_sequences

Training sequences: 45795980
Expected training sequences: 45795980
Actual training sequences: 45795980


In [12]:
validation_dataset = RetailSequenceDataset(
    data=scaled_all,
    start_target=train_end,
    end_target=validation_end,
    lookback=LOOKBACK
)

print(
    "Validation sequences:",
    len(validation_dataset)
)

Validation sequences: 5823590


In [13]:
test_dataset = RetailSequenceDataset(
    data=scaled_all,
    start_target=validation_end,
    end_target=test_end,
    lookback=LOOKBACK
)

print(
    "Test sequences:",
    len(test_dataset)
)

Test sequences: 5854080


In [14]:
X_sample, y_sample = train_dataset[0]

print("X sample shape:", X_sample.shape)
print("y sample shape:", y_sample.shape)

print("X sample:")
print(X_sample.squeeze(-1))

print("y sample:")
print(y_sample)

X sample shape: torch.Size([28, 1])
y sample shape: torch.Size([])
X sample:
tensor([-0.2758, -0.2758, -0.2758, -0.2758, -0.2758, -0.2758, -0.2758, -0.2758,
        -0.2758, -0.2758, -0.2758, -0.2758, -0.2758, -0.2758, -0.2758, -0.2758,
        -0.2758, -0.2758, -0.2758, -0.2758, -0.2758, -0.2758, -0.2758, -0.2758,
        -0.2758, -0.2758, -0.2758, -0.2758])
y sample:
tensor(-0.2758)


In [15]:
first_target = LOOKBACK

first_input_start = (
    first_target -
    LOOKBACK
)

first_input_end = first_target

print(
    "Input indexes:",
    first_input_start,
    "to",
    first_input_end - 1
)

print(
    "Target index:",
    first_target
)

Input indexes: 0 to 27
Target index: 28


In [16]:
X_val_sample, y_val_sample = validation_dataset[0]

print(
    "Validation first target index:",
    train_end
)

print(
    "Validation input indexes:",
    train_end - LOOKBACK,
    "to",
    train_end - 1
)

Validation first target index: 1530
Validation input indexes: 1502 to 1529


In [17]:
X_test_sample, y_test_sample = test_dataset[0]

print(
    "Test first target index:",
    validation_end
)

print(
    "Test input indexes:",
    validation_end - LOOKBACK,
    "to",
    validation_end - 1
)

Test first target index: 1721
Test input indexes: 1693 to 1720


In [18]:
from torch.utils.data import DataLoader

BATCH_SIZE = 256

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

In [19]:
X_batch, y_batch = next(
    iter(train_loader)
)

print("X batch shape:", X_batch.shape)
print("y batch shape:", y_batch.shape)

X batch shape: torch.Size([256, 28, 1])
y batch shape: torch.Size([256])


In [20]:
print("Training dataset length:", len(train_dataset))
print("Validation dataset length:", len(validation_dataset))
print("Test dataset length:", len(test_dataset))

expected_validation_sequences = (
    len(sales_clean) *
    (validation_end - train_end)
)

expected_test_sequences = (
    len(sales_clean) *
    (test_end - validation_end)
)

assert len(validation_dataset) == (
    expected_validation_sequences
)

assert len(test_dataset) == (
    expected_test_sequences
)

print("Dataset lengths validated.")

Training dataset length: 45795980
Validation dataset length: 5823590
Test dataset length: 5854080
Dataset lengths validated.


In [21]:
def validate_dataset_boundaries(
    dataset,
    first_target,
    last_target
):
    X_first, y_first = dataset[0]

    last_index = len(dataset) - 1

    X_last, y_last = dataset[last_index]

    first_input_length = X_first.shape[0]
    last_input_length = X_last.shape[0]

    checks = {
        "first_input_length_correct":
            first_input_length == LOOKBACK,

        "last_input_length_correct":
            last_input_length == LOOKBACK,

        "first_target_valid":
            first_target >= LOOKBACK,

        "last_target_valid":
            last_target < test_end
    }

    checks["valid"] = all(checks.values())

    return checks

train_boundary_check = validate_dataset_boundaries(
    train_dataset,
    LOOKBACK,
    train_end - 1
)

validation_boundary_check = validate_dataset_boundaries(
    validation_dataset,
    train_end,
    validation_end - 1
)

test_boundary_check = validate_dataset_boundaries(
    test_dataset,
    validation_end,
    test_end - 1
)

print("Training:", train_boundary_check)
print("Validation:", validation_boundary_check)
print("Test:", test_boundary_check)

Training: {'first_input_length_correct': True, 'last_input_length_correct': True, 'first_target_valid': True, 'last_target_valid': True, 'valid': True}
Validation: {'first_input_length_correct': True, 'last_input_length_correct': True, 'first_target_valid': True, 'last_target_valid': True, 'valid': True}
Test: {'first_input_length_correct': True, 'last_input_length_correct': True, 'first_target_valid': True, 'last_target_valid': True, 'valid': True}


In [22]:
sequence_summary = pd.DataFrame({
    "dataset": [
        "train",
        "validation",
        "test"
    ],
    "target_start_index": [
        LOOKBACK,
        train_end,
        validation_end
    ],
    "target_end_index": [
        train_end - 1,
        validation_end - 1,
        test_end - 1
    ],
    "number_of_target_days": [
        train_end - LOOKBACK,
        validation_end - train_end,
        test_end - validation_end
    ],
    "number_of_series": [
        len(sales_clean)
    ] * 3,
    "lookback_days": [
        LOOKBACK
    ] * 3,
    "features_per_timestep": [
        1
    ] * 3
})

sequence_summary.to_csv(
    results_path / "sequence_generation_summary.csv",
    index=False
)
display(sequence_summary)

,dataset,target_start_index,target_end_index,number_of_target_days,number_of_series,lookback_days,features_per_timestep
0,train,28,1529,1502,30490,28,1
1,validation,1530,1720,191,30490,28,1
2,test,1721,1912,192,30490,28,1


In [23]:
sequence_validation = pd.DataFrame({
    "check": [
        "training_boundary",
        "validation_boundary",
        "test_boundary",
        "training_dataset_length",
        "validation_dataset_length",
        "test_dataset_length"
    ],
    "passed": [
        train_boundary_check["valid"],
        validation_boundary_check["valid"],
        test_boundary_check["valid"],
        len(train_dataset) == expected_train_sequences,
        len(validation_dataset) == expected_validation_sequences,
        len(test_dataset) == expected_test_sequences
    ]
})

sequence_validation.to_csv(
    results_path / "sequence_validation.csv",
    index=False
)
display(sequence_validation)

,check,passed
0,training_boundary,True
1,validation_boundary,True
2,test_boundary,True
3,training_dataset_length,True
4,validation_dataset_length,True
5,test_dataset_length,True


In [24]:
assert train_boundary_check["valid"]
assert validation_boundary_check["valid"]
assert test_boundary_check["valid"]

assert len(train_dataset) == expected_train_sequences
assert len(validation_dataset) == expected_validation_sequences
assert len(test_dataset) == expected_test_sequences

X_batch, y_batch = next(
    iter(train_loader)
)

assert X_batch.shape == (
    BATCH_SIZE,
    LOOKBACK,
    1
)

assert y_batch.shape == (
    BATCH_SIZE,
)

print("======================================")
print("SEQUENCE GENERATION VALIDATION PASSED")
print("======================================")

SEQUENCE GENERATION VALIDATION PASSED


## Final Sequence Representation

The forecasting problem is represented as supervised learning:

    X(t) = [d(t-28), ..., d(t-2), d(t-1)]

    y(t) = d(t)

where:

- 28 historical observations form the input.
- The following day's demand is the prediction target.
- One numerical demand feature is provided at each time step.

The resulting neural-network input shape is:

    (batch_size, 28, 1)

This representation is shared across the four architectures so that model
comparisons are based on architectural differences rather than different
input preparation procedures.

## Memory Efficiency

The complete M5 dataset contains 30,490 retail series and thousands of
daily observations. Materializing every sliding window as a NumPy array
would require excessive memory.

Therefore, the implementation uses a PyTorch Dataset that generates each
28-day sequence lazily when it is requested by a DataLoader.

This provides:

- Memory-efficient training
- Consistent sequence construction
- Mini-batch training
- GPU-compatible tensors
- Identical sequence definitions across all models

## Temporal Integrity

Training targets occur before validation targets, and validation targets
occur before test targets.

For validation and test predictions, historical observations immediately
before the target are allowed as input context because they would be
available at prediction time.

Future target observations are never included in the corresponding input
window.